In [ ]:
import numpy as np
from numpy.linalg import eigvals, eigvalsh, inv, norm, solve
import matplotlib.pyplot as plt

In [ ]:
n = 4
A = np.array([
    [4.0, 1.5, 0.5, 0.0],
    [1.5, 3.0, 0.0, 0.0],
    [0.5, 0.0, 2.0, 0.0],
    [0.0, 0.0, 0.0, -2.0]
])
b = np.array([1.0, -2.0, 1.5, -1.0])
x0 = np.array([0.5, 0.5, 0.5, 0.5])
r = 1.0

print(f"  Размерность: n = {n}")
print(f"  Матрица A:\n{A}")
print(f"  Вектор b: {b}")
print(f"  Центр сферы x0: {x0}")
print(f"  Радиус сферы r: {r}")


eigenvalues = eigvals(A)
print(f"\n Собственные значения: {np.round(eigenvalues, 4)}")

pos = sum(eigenvalues > 1e-10)
neg = sum(eigenvalues < -1e-10)
print(f"  Положительных: {pos}, Отрицательных: {neg}")

print(f"  det(A) = {np.linalg.det(A):.6f}")
if abs(np.linalg.det(A)) > 1e-10:
    print("  => матрица A невырожденная")

  Размерность: n = 4
  Матрица A:
[[ 4.   1.5  0.5  0. ]
 [ 1.5  3.   0.   0. ]
 [ 0.5  0.   2.   0. ]
 [ 0.   0.   0.  -2. ]]
  Вектор b: [ 1.  -2.   1.5 -1. ]
  Центр сферы x0: [0.5 0.5 0.5 0.5]
  Радиус сферы r: 1.0

 Собственные значения: [ 5.1341  1.6427  2.2232 -2.    ]
  Положительных: 3, Отрицательных: 1
  det(A) = -37.500000
  => матрица A невырожденная


In [ ]:
def f(x):
    return 0.5 * x @ A @ x + b @ x

def grad_f(x):
    return A @ x + b

def g(x):
    return np.linalg.norm(x - x0) - r

In [ ]:
x_uncond = -inv(A) @ b
f_uncond = f(x_uncond)
dist_uncond = np.linalg.norm(x_uncond - x0)

print(f"  x_uncond = {np.round(x_uncond, 6)}")
print(f"  f(x_uncond) = {f_uncond:.6f}")
print(f"  ||x_uncond - x0|| = {dist_uncond:.6f}")
print(f"  r = {r}")

if dist_uncond <= r:
    print("  Безусловный минимум внутри сферы это и есть решение")
    constraint_active = False
else:
    print(f" Безусловный минимум за сферой ({dist_uncond:.4f} > {r})")
    print(" То есть ищем минимум на границе сферы")
    constraint_active = True

  x_uncond = [-0.52      0.926667 -0.62     -0.5     ]
  f(x_uncond) = -1.401667
  ||x_uncond - x0|| = 1.864630
  r = 1.0
 Безусловный минимум за сферой (1.8646 > 1.0)
 То есть ищем минимум на границе сферы


In [ ]:
if constraint_active:
    c_vec = A @ x0 + b

    def F_lambda(lmbda):
        """F(λ) = ||y(λ)||² - r²"""
        M = A + lmbda * np.eye(n)
        try:
            y = solve(M, -c_vec)
            return np.dot(y, y) - r**2
        except np.linalg.LinAlgError:
            return np.inf

    def dF_lambda(lmbda):
        """F'(λ) = -2·yᵀ·(A + λI)⁻¹·y"""
        M = A + lmbda * np.eye(n)
        try:
            y = solve(M, -c_vec)
            z = solve(M, y)
            return -2.0 * np.dot(y, z)
        except np.linalg.LinAlgError:
            return 0.0

    print(f"\n  Метод Ньютона (касательных) для поиска λ:")
    print(f"  {'Итер.':<6} {'λ_k':<15} {'F(λ_k)':<12} {'F\'(λ_k)':<12}")

    min_eig = np.min(eigvalsh(A))
    lam = max(0.1, -min_eig + 1.0)

    tol = 1e-12
    max_iter = 100

    #метод ньютона
    for k in range(max_iter):
        F_val = F_lambda(lam)
        dF_val = dF_lambda(lam)

        print(f"  {k+1:<6} {lam:<15.10f} {F_val:<12.6e} {dF_val:<12.6e}")

        if abs(dF_val) < 1e-14:
            print("  Производная близка к нулю")
            break

        lam_new = lam - F_val / dF_val

        if lam_new <= -min_eig + 1e-6:
            lam_new = lam * 0.5

        if abs(lam_new - lam) < tol:
            lam = lam_new
            print(f"\n Сходимость достигнута за {k+1} итераций")
            break

        lam = lam_new

    lambda_opt = lam
    print(f"\n  Найденный множитель Лагранжа: λ = {lambda_opt:.10f}")

    M_opt = A + lambda_opt * np.eye(n)
    y_opt = solve(M_opt, -c_vec)
    x_opt = y_opt + x0
    f_opt = f(x_opt)

    print(f"\n  Оптимальная точка: x* = {np.round(x_opt, 6)}")
    print(f"  f(x*) = {f_opt:.10f}")
    print(f"  ||x* - x0|| = {np.linalg.norm(x_opt - x0):.10f}")



  Метод Ньютона (касательных) для поиска λ:
  Итер.  λ_k             F(λ_k)       F'(λ_k)     
  1      3.0000000000    3.564140e+00 -8.186998e+00
  2      3.4353415435    1.432590e+00 -2.856005e+00
  3      3.9369477293    4.897770e-01 -1.220664e+00
  4      4.3381858983    1.111036e-01 -7.268612e-01
  5      4.4910398088    9.096658e-03 -6.124914e-01
  6      4.5058917043    7.239873e-05 -6.027790e-01
  7      4.5060118126    4.662984e-09 -6.027013e-01
  8      4.5060118203    0.000000e+00 -6.027013e-01

 Сходимость достигнута за 8 итераций

  Найденный множитель Лагранжа: λ = 4.5060118203

  Оптимальная точка: x* = [0.042265 0.558167 0.112492 1.298081]
  f(x*) = -3.3671089001
  ||x* - x0|| = 1.0000000000


In [ ]:
def newton_method_kkt(x_start, lambda_start=1.0, max_iter=100, tol=1e-10, verbose=False):

    x = x_start.copy()
    lam = lambda_start

    if verbose:
        print(f"  Начальная точка: x = {np.round(x, 4)}, λ = {lam:.4f}")
        print(f"  {'Итер.':<6} {'f(x)':<12} {'||x-x0||':<10} {'λ':<12} {'||Δ||':<10}")

    for k in range(max_iter):
        # (A + λI)x + b - λx0 = 0
        residual1 = (A + lam * np.eye(n)) @ x + b - lam * x0

        # ||x - x0||² - r² = 0
        residual2 = np.dot(x - x0, x - x0) - r**2

        J = np.zeros((n+1, n+1))
        J[:n, :n] = A + lam * np.eye(n)
        J[:n, n] = x - x0
        J[n, :n] = 2 * (x - x0)
        J[n, n] = 0

        residual = np.concatenate([residual1, [residual2]])

        try:
            delta = solve(J, -residual)
            delta_x = delta[:n]
            delta_lam = delta[n]
        except np.linalg.LinAlgError:
            if verbose:
                print(f"  Матрица Якобиана вырождена на итерации {k+1}")
            break

        x_new = x + delta_x
        lam_new = lam + delta_lam

        # Ограничение λ ≥ 0
        if lam_new < 0:
            lam_new = 0.0

        delta_norm = np.linalg.norm(delta)
        f_val = f(x_new)
        constraint_val = np.linalg.norm(x_new - x0)

        if verbose:
            print(f"  {k+1:<6} {f_val:<12.6f} {constraint_val:<10.6f} {lam_new:<12.6f} {delta_norm:<10.2e}")

        if delta_norm < tol:
            if verbose:
                print(f"  Сходимость за {k+1} итераций")
            return x_new, lam_new, k+1, True

        x = x_new
        lam = lam_new

    return x, lam, max_iter, False

np.random.seed(42)

# 4 точки внутри области
inside_points = []
for i in range(4):
    v = np.random.randn(n)
    v = v / np.linalg.norm(v) * r * 0.5 * np.random.uniform(0.3, 0.9)
    inside_points.append(x0 + v)

# 4 точки на границе
boundary_points = []
for i in range(4):
    v = np.random.randn(n)
    v = v / np.linalg.norm(v) * r
    boundary_points.append(x0 + v)

# 4 точки вне области
outside_points = []
for i in range(4):
    v = np.random.randn(n)
    v = v / np.linalg.norm(v) * r * np.random.uniform(1.1, 2.0)
    outside_points.append(x0 + v)

all_points = {
    'Внутри области': inside_points,
    'На границе': boundary_points,
    'Вне области': outside_points
}

results = []
experiment_num = 1

for region, points in all_points.items():

    for i, x_start in enumerate(points):
        print(f"\n  Эксперимент {experiment_num}: {region} {i+1}")
        print(f"  Начальная точка: x₀ = {np.round(x_start, 4)}")
        print(f"  ||x₀ - x0|| = {np.linalg.norm(x_start - x0):.4f}")

        x_result, lam_result, iters, converged = newton_method_kkt(
            x_start, lambda_start=1.0, max_iter=50, tol=1e-10, verbose=True
        )

        f_result = f(x_result)
        constraint_result = np.linalg.norm(x_result - x0)

        results.append({
            'experiment': experiment_num,
            'region': region,
            'x_start': x_start.copy(),
            'x_opt': x_result.copy(),
            'lambda': lam_result,
            'f_opt': f_result,
            'constraint': constraint_result,
            'iterations': iters,
            'converged': converged
        })

        print(f"\n  Результат:")
        print(f"    x* = {np.round(x_result, 6)}")
        print(f"    λ = {lam_result:.10f}")
        print(f"    f(x*) = {f_result:.10f}")
        print(f"    ||x* - x0|| = {constraint_result:.10f}")
        print(f"    Итераций: {iters}")
        print(f"    Сходимость: {'Yes' if converged else 'NO'}")

        experiment_num += 1


  Эксперимент 1: Внутри области 1
  Начальная точка: x₀ = [0.5564 0.4843 0.5735 0.6729]
  ||x₀ - x0|| = 0.1968
  Начальная точка: x = [0.5564 0.4843 0.5735 0.6729], λ = 1.0000
  Итер.  f(x)         ||x-x0||   λ            ||Δ||     
  1      -27.263810   4.599970   36.368784    3.57e+01  
  2      -11.042501   2.409135   18.705940    1.78e+01  
  3      -5.300536    1.413333   9.905350     8.86e+00  
  4      -3.643288    1.065788   5.794240     4.13e+00  
  5      -3.391607    1.005633   4.598254     1.20e+00  
  6      -3.368003    1.000199   4.506590     9.38e-02  
  7      -3.367109    1.000000   4.506012     7.23e-04  
  8      -3.367109    1.000000   4.506012     1.60e-07  
  9      -3.367109    1.000000   4.506012     3.81e-14  
  Сходимость за 9 итераций

  Результат:
    x* = [0.042265 0.558167 0.112492 1.298081]
    λ = 4.5060118203
    f(x*) = -3.3671089001
    ||x* - x0|| = 1.0000000000
    Итераций: 9
    Сходимость: Yes

  Эксперимент 2: Внутри области 2
  Начальная точк

In [ ]:
print(f"\n{'Эксп.':<6} {'Область':<15} {'f(x*)':<15} {'λ':<12} {'||x*-x0||':<12} {'Итер.':<6} {'Сход.':<6}")
print(f"{'─'*75}")

for res in results:
    print(f"{res['experiment']:<6} {res['region']:<15} {res['f_opt']:<15.8f} {res['lambda']:<12.6f} {res['constraint']:<12.6f} {res['iterations']:<6} {'Yes' if res['converged'] else 'No':<6}")

f_values = [res['f_opt'] for res in results]
x_values = [res['x_opt'] for res in results]

f_mean = np.mean(f_values)
f_std = np.std(f_values)
x_mean = np.mean(x_values, axis=0)
x_std = np.std(x_values, axis=0)

if f_std < 1e-6:
    print(f"  Все эксперименты сошлись к одному решению")
else:
    print(f"  Эксперименты дали разные результаты")


print(f"\n Минимум функции: f(x*) = {f_opt:.7f}")
print(f"  Достигается в точке: x* = {np.round(x_opt, 6)}")
print(f"  Множитель Лагранжа: λ = {lambda_opt:.10f}")
print(f"  Ограничение активно: ||x* - x0|| = {np.linalg.norm(x_opt - x0):.10f} = r")


Эксп.  Область         f(x*)           λ            ||x*-x0||    Итер.  Сход. 
───────────────────────────────────────────────────────────────────────────
1      Внутри области  -3.36710890     4.506012     1.000000     9      Yes   
2      Внутри области  -3.36710890     4.506012     1.000000     12     Yes   
3      Внутри области  -1.22585897     0.511812     1.312212     50     No    
4      Внутри области  -3.36710890     4.506012     1.000000     30     Yes   
5      На границе      -3.36710890     4.506012     1.000000     38     Yes   
6      На границе      -0.16259068     2.652193     1.831868     50     No    
7      На границе      -3.36710890     4.506012     1.000000     17     Yes   
8      На границе      -3.36764969     4.502691     1.000120     50     No    
9      Вне области     -3.36710890     4.506012     1.000000     26     Yes   
10     Вне области     -3.36710890     4.506012     1.000000     44     Yes   
11     Вне области     -3.36710890     4.506012     1.